# 01 · 승인 확률과 잔차 분포

작은 이산 분포 $p$, $q$에서 speculative sampling 한 단계가 $p$를 복원하는지 확인한다. 실제 Transformer 결과를 재현하지 않는 toy reproduction이다.

**학습 목표**: 승인 질량과 잔차 분포의 합이 목표 분포 $p$를 복원하는 이유를 계산과 Monte Carlo로 확인한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `random`, `collections`만 사용하며 외부 패키지는 없다.

In [ ]:
# Random 객체를 함수에 전달해 확률 알고리즘과 재현 가능한 seed를 분리한다.
import random
from collections import Counter

vocab = ('A', 'B', 'C')
p = {'A': 0.50, 'B': 0.30, 'C': 0.20}
q = {'A': 0.20, 'B': 0.50, 'C': 0.30}

def draw(dist, rng):
    point, cumulative = rng.random(), 0.0
    for token, probability in dist.items():
        cumulative += probability
        if point <= cumulative:
            return token
    return next(reversed(dist))

def residual(p_dist, q_dist):
    raw = {x: max(0.0, p_dist[x] - q_dist[x]) for x in p_dist}
    total = sum(raw.values())
    return {x: value / total for x, value in raw.items()}

def speculative_one(p_dist, q_dist, rng):
    candidate = draw(q_dist, rng)
    if rng.random() <= min(1.0, p_dist[candidate] / q_dist[candidate]):
        return candidate
    return draw(residual(p_dist, q_dist), rng)


In [ ]:
overlap = sum(min(p[x], q[x]) for x in vocab)
print('예상 승인률 alpha =', round(overlap, 3))
print('거절 시 잔차 분포 =', residual(p, q))

rng = random.Random(7)
n = 100_000
counts = Counter(speculative_one(p, q, rng) for _ in range(n))
empirical = {x: counts[x] / n for x in vocab}
print('target   =', p)
print('empirical=', {x: round(v, 4) for x, v in empirical.items()})
assert max(abs(empirical[x] - p[x]) for x in vocab) < 0.01


초안 승인 질량 $\min(p,q)$와 거절 후 $[p-q]_+$ 질량을 합치면 목표 $p$가 된다. residual 보정을 생략하고 목표 분포에서 무조건 다시 뽑으면 정확성 증명이 깨진다.